# Visualization and Analysis

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/douglasrizzo/catsim/blob/main/notebooks/05_visualization_and_analysis.ipynb)

`catsim.plot` contains the high-level plots that most users need when they
are exploring an item bank or interpreting a CAT simulation. It needs
matplotlib, which the plot extra installs:

```bash
pip install -U 'catsim[plot]'
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from catsim import plot
from catsim.engine import CatEngine
from catsim.estimation import NumericalSearchEstimator
from catsim.initialization import UniformInitializer
from catsim.item_bank import ItemBank
from catsim.selection import MaxInfoSelector
from catsim.simulation import SimulationRunner
from catsim.stopping import MaxLengthStopper

In [ ]:
item_bank = ItemBank.generate(120, rng=41)
engine = CatEngine(
  item_bank=item_bank,
  initializer=UniformInitializer(),
  selector=MaxInfoSelector(),
  estimator=NumericalSearchEstimator(),
  stopper=MaxLengthStopper(12),
)
true_thetas = np.random.default_rng(3).normal(0, 1, 25)
result = SimulationRunner(engine).run(true_thetas, rng=101)

## Plot one item

The top panel is the item characteristic curve: the probability of a correct
response rises from the guessing parameter $c$ to the upper asymptote $d$, most
steeply around the difficulty $b$. The middle panel is the item information
curve, and the dot marks its peak, a little above $b$ for an item with a
guessing parameter. The bottom panel draws both, the information against the
right y axis.

In [ ]:
item = item_bank.items[5]
fig, axes = plt.subplots(3, 1, figsize=(7, 12))
plot.item_curve(item, kind=plot.PlotType.ICC, ax=axes[0])
plot.item_curve(item, kind=plot.PlotType.IIC, ax=axes[1])
plot.item_curve(item, kind=plot.PlotType.BOTH, ax=axes[2])
plt.tight_layout()
plt.show()
plt.close("all")

## Visualize one session trace

The left y axis is the ability scale: the estimate $\hat{\theta}$ after each
item, with a band of one standard error on each side, the true $\theta$
(dashed) and the difficulty of each administered item. After the 12 items the
stopper allows, the estimate has come close to the true ability, which lies
inside the band. The right y axis shows the standard error the estimator
recorded after each item: it shrinks as the responses accumulate, and the band
narrows with it. The initial estimate, at 0 items, has no standard error yet.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
plot.session_progress(result.sessions[0], result.item_bank, ax=ax)
plt.tight_layout()
plt.show()
plt.close("all")

## Visualize exposure patterns

The upper panel shows each item's exposure rate against its discrimination $a$:
maximum-information selection never administers the least discriminating items
and gives the most discriminating ones to most examinees. The histogram below
counts items by exposure rate: most items are never administered, while a few
go to most examinees.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 8))
plot.item_exposure(
  result.exposure_rates, item_bank=result.item_bank, par="a", ax=axes[0], title="Exposure by Discrimination"
)
plot.item_exposure(result.exposure_rates, hist=True, ax=axes[1], title="Exposure Histogram")
plt.tight_layout()
plt.show()
plt.close("all")

## Reporting guidance

- Item curves help explain the meaning of item parameters.
- Session progress plots show how the CAT converged for one examinee.
- Exposure plots help evaluate item security and distribution.

These figures are descriptive. They are useful for communicating results,
but they should be interpreted together with design choices such as test
length, stopper settings, and item-bank calibration quality.